# Exact extremum with native FlagQuantum

Preserve both original tutorial examples and their order. Use coherent quantum values, local CPU execution and exact named results; seeded local samples replace the upstream cloud sampler. No hardware result is claimed.


In [ ]:
from collections import Counter
from fractions import Fraction
from random import Random
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc, synthesize


## Minimum

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def minimum(a: Output[QNum], b: Output[Annotated[QNum, 3]], res: Output[QNum]) -> None:
    a |= 4
    allocate(b)
    hadamard_transform(b)
    res |= a if a < b else b


In [ ]:
minimum_program = synthesize(minimum)
minimum_result = fq.run(minimum_program)
minimum_probabilities = dict(minimum_program.probabilities(minimum_result, registers=('a', 'b', 'res')))
expected = {(4, b, min(4, b)): 1/8 for b in range(8)}
assert minimum_probabilities.keys() == expected.keys()
assert all(abs(minimum_probabilities[key]-value) < 1e-12 for key,value in expected.items())
minimum_counts = Counter(Random(17).choices(list(minimum_probabilities), weights=list(minimum_probabilities.values()), k=1000))
assert sum(minimum_counts.values()) == 1000
print("Probabilities:", minimum_probabilities)
print("Counts:", dict(minimum_counts))
print("Resources:", minimum_program.report["resources"])


## Maximum

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def maximum(a: Output[Annotated[QNum, 3]], res: Output[QNum]) -> None:
    allocate(a)
    hadamard_transform(a)
    res |= synth.max(3.5, a)


In [ ]:
maximum_program = synthesize(maximum)
maximum_result = fq.run(maximum_program)
maximum_probabilities = dict(maximum_program.numeric_probabilities(maximum_result, registers=('a', 'res')))
expected = {(a, max(Fraction(7, 2), a)): 1/8 for a in range(8)}
assert maximum_probabilities.keys() == expected.keys()
assert all(abs(maximum_probabilities[key]-value) < 1e-12 for key,value in expected.items())
maximum_counts = Counter(Random(17).choices(list(maximum_probabilities), weights=list(maximum_probabilities.values()), k=1000))
assert sum(maximum_counts.values()) == 1000
print("Probabilities:", maximum_probabilities)
print("Counts:", dict(maximum_counts))
print("Resources:", maximum_program.report["resources"])


## What is checked

These examples keep their original inputs, including exact binary coefficient 3.5 where present. Independent Circuit acceptance must additionally check the complete complex state and work cleanup; probabilities and samples alone do not certify phase.
